## Load the corpus

We use a labeled Movie Review Dataset to implement a simple supervised machine learning approach. The dataset contains 5,331 positive and 5,331 negative processed sentences from Rotten Tomatoes movie reviews.
Our goal is to train a classifier that can predict whether a sentence is positive or negative.

This data was first used in Bo Pang and Lillian Lee, ``Seeing stars: Exploiting class relationships for sentiment categorization with respect to rating scales.'', Proceedings of the ACL, 2005. Please find more information on the dataset here.
https://huggingface.co/datasets/rotten_tomatoes

Data labeling: "the movie-review “snippets” (a striking extract usually one sentence long) downloaded from www.rottentomatoes.com; each
snippet was labeled with its source review’s label
(positive or negative) as provided by Rotten Tomatoes." (Pang & Lee, 2005) p.119)

In [ ]:
# !pip install scikit-learn nltk shap lime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder


from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_predict, cross_val_score

from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve
from sklearn.metrics import classification_report, accuracy_score, auc, average_precision_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk

from collections import defaultdict
import math

import shap, lime
from lime.lime_text import LimeTextExplainer



In [ ]:
# Download required NLTK data files
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [ ]:


# Load the dataset with specified encoding
reviews = pd.read_csv("reviews.csv", encoding='latin1')  # Use 'latin1' encoding for non-UTF-8 characters

# Inspect the data
print(reviews.head())  # Display the first few rows
print(reviews.info())  # General information about the dataset


# Inspecting the data

In [ ]:
# View column names
print(reviews.columns)

# Rename the column 'value' to 'text'
reviews.rename(columns={'value': 'text'}, inplace=True)

# Check the class of the 'text' column
print(type(reviews['text']))

# Check the distribution of the outcome variable ('polarity')
print(reviews['polarity'].value_counts())

# Check the class of the 'polarity' column
print(type(reviews['polarity']))


In [ ]:
# Keep the same stratified split throughout the notebook.
from sklearn.model_selection import train_test_split
reviews['id_numeric'] = range(1, len(reviews) + 1)
id_train, id_test = train_test_split(
    reviews['id_numeric'], test_size=0.2, random_state=2026,
    stratify=reviews['polarity'])
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
def preprocess(text):
    tokens = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in tokens
            if word.isalpha() and word not in stop_words]


In [ ]:

# Apply preprocessing to the 'text' column
reviews['processed_text'] = reviews['text'].apply(preprocess)


In [ ]:
reviews

In [ ]:
# Learn the vocabulary and frequency filter from training documents only.
train_mask = reviews['id_numeric'].isin(id_train).to_numpy()
test_mask = reviews['id_numeric'].isin(id_test).to_numpy()
vectorizer = CountVectorizer(analyzer=lambda x: x, min_df=2)
dfm_train = vectorizer.fit_transform(reviews.loc[train_mask, 'processed_text'])
dfm_test = vectorizer.transform(reviews.loc[test_mask, 'processed_text'])
# For demonstrations that need a single matrix, preserve the original row order.
dfm_reviews = vectorizer.transform(reviews['processed_text'])
dfm_trimmed = dfm_reviews


In [ ]:
dfm_train

In [ ]:
dfm_test

In [ ]:
# Summary of results
print("DFM Shape (Full):", dfm_reviews.shape)
print("DFM Shape (Trimmed):", dfm_trimmed.shape)
print("Training Set Shape:", dfm_train.shape)
print("Test Set Shape:", dfm_test.shape)



# Training


In [ ]:

# Encode the target variable (polarity) as numerical values
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(reviews.loc[train_mask, 'polarity'])
y_test = label_encoder.transform(reviews.loc[test_mask, 'polarity'])


In [ ]:
y_test

In [ ]:

# Create a Multinomial Naive Bayes model
# Scikit-learn's MultinomialNB uses term frequencies directly
model_nb = MultinomialNB()

# model_nb = MultinomialNB(class_prior=[0.5, 0.5])  # Example: Equal priors


# Fit the model on the training data
model_nb.fit(dfm_train, y_train)

# Model summary
print("Classes:", label_encoder.classes_)
print("Class Log Prior (from docfreq):", model_nb.class_log_prior_)


In [ ]:
np.log(0.5)

In [ ]:
# Encode the target variable (polarity) as numerical values
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(reviews.loc[train_mask, 'polarity'])
y_test = label_encoder.transform(reviews.loc[test_mask, 'polarity'])

# Create and train the SVM model
model_svm = LinearSVC()  # Linear kernel SVM suitable for high-dimensional sparse data
model_svm.fit(dfm_train, y_train)

# Model summary
print("Classes:", label_encoder.classes_)

# Predict for the test set (NB)


In [ ]:
# Predict for the test set using Naive Bayes
y_pred_nb = model_nb.predict(dfm_test)

# Decode predictions back to original labels
y_pred_nb_decoded = label_encoder.inverse_transform(y_pred_nb)

# Summary of predictions
print("Naive Bayes Predictions Summary:")
print(pd.Series(y_pred_nb_decoded).value_counts())


In [ ]:
# Summary of the Naive Bayes model
print("Naive Bayes Model Class Log Priors:")
print(model_nb.class_log_prior_)

print("Naive Bayes Model Feature Log Probabilities:")
print(model_nb.feature_log_prob_)  # Log probabilities of features per class


# Predict for test set (SVM)

In [ ]:
# Predict for the test set using SVM
y_pred_svm = model_svm.predict(dfm_test)

# Decode predictions back to original labels
y_pred_svm_decoded = label_encoder.inverse_transform(y_pred_svm)

# Summary of predictions
print("SVM Predictions Summary:")
print(pd.Series(y_pred_svm_decoded).value_counts())


# Precision / Recall / F1 Scores

$$
\text{Precision} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Positives (FP)}}
$$

$$
\text{Recall} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Negatives (FN)}}
$$


$$
\text{F1-Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$


$$
\text{Macro F1-Score} = \frac{1}{C} \sum_{i=1}^{C} F_1^i
$$



In [ ]:
# Assume y_test is the ground truth and predictions are already made
# y_test: True labels for the test set
# y_pred_nb: Predictions from Naive Bayes
# y_pred_svm: Predictions from SVM

def calculate_metrics(y_true, y_pred):
    """
    Args:
        y_true (list): True labels.
        y_pred (list): Predicted labels.
    Returns:
        dict: Metrics for each class.
    """
    # Get unique classes
    classes = set(y_true)
    metrics = {}

    for cls in classes:
        # True Positives (TP)
        tp = sum((y_pred[i] == cls) and (y_true[i] == cls) for i in range(len(y_true)))
        # False Positives (FP)
        fp = sum((y_pred[i] == cls) and (y_true[i] != cls) for i in range(len(y_true)))
        # False Negatives (FN)
        fn = sum((y_pred[i] != cls) and (y_true[i] == cls) for i in range(len(y_true)))

        # Precision, Recall, F1-Score
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        # Store metrics for the class
        metrics[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1_score
        }

    return metrics


In [ ]:

# Decode y_test to original labels (if necessary)
y_test_decoded = label_encoder.inverse_transform(y_test)

# Decode predictions to original labels
y_pred_nb_decoded = label_encoder.inverse_transform(y_pred_nb)
y_pred_svm_decoded = label_encoder.inverse_transform(y_pred_svm)

# Calculate metrics for Naive Bayes
nb_metrics = calculate_metrics(y_test_decoded, y_pred_nb_decoded)

# Calculate metrics for SVM
svm_metrics = calculate_metrics(y_test_decoded, y_pred_svm_decoded)

# Display results
print("Metrics for Naive Bayes:")
for cls, scores in nb_metrics.items():
    print(f"Class: {cls}")
    print(f"  Precision: {scores['Precision']:.2f}")
    print(f"  Recall: {scores['Recall']:.2f}")
    print(f"  F1-Score: {scores['F1-Score']:.2f}")

print("\nMetrics for SVM:")
for cls, scores in svm_metrics.items():
    print(f"Class: {cls}")
    print(f"  Precision: {scores['Precision']:.2f}")
    print(f"  Recall: {scores['Recall']:.2f}")
    print(f"  F1-Score: {scores['F1-Score']:.2f}")

# Accuracy

In [ ]:
def compute_accuracy(y_true, y_pred):
    """
    Compute accuracy as the ratio of correctly predicted instances to the total instances.
    Args:
        y_true (list): True labels.
        y_pred (list): Predicted labels.
    Returns:
        float: Accuracy value.
    """
    # Create a confusion matrix
    conf_mat = pd.crosstab(pd.Series(y_true, name="Actual"), pd.Series(y_pred, name="Predicted"))
    
    # Extract diagonal (correctly predicted instances)
    correct_predictions = conf_mat.to_numpy().diagonal().sum()
    total_instances = conf_mat.to_numpy().sum()
    
    # Calculate accuracy
    accuracy = correct_predictions / total_instances
    return accuracy

# Example usage with Naive Bayes predictions
accuracy_nb = compute_accuracy(y_test_decoded, y_pred_nb_decoded)
print(f"Naive Bayes Accuracy: {accuracy_nb:.2f}")

# Example usage with SVM predictions
accuracy_svm = compute_accuracy(y_test_decoded, y_pred_svm_decoded)
print(f"SVM Accuracy: {accuracy_svm:.2f}")


In [ ]:


def plot_confusion_matrix(y_true, y_pred, color="Blues"):
    """
    Args:
        y_true (list): True labels.
        y_pred (list): Predicted labels.
        color (str): Color scheme for the heatmap.
    Returns:
        None
    """
    # Create a confusion matrix
    conf_mat = pd.crosstab(pd.Series(y_true, name="Actual"), pd.Series(y_pred, name="Predicted"))
    
    # Normalize rows to proportions
    conf_mat_normalized = conf_mat.div(conf_mat.sum(axis=1), axis=0)
    
    # Plot heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_mat_normalized, annot=True, fmt=".2f", cmap=color, cbar=True, linewidths=1, linecolor="gray")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Labels")
    plt.ylabel("True Labels")
    plt.show()

# Example usage with Naive Bayes predictions
plot_confusion_matrix(y_test_decoded, y_pred_nb_decoded, color="Blues")

# Example usage with SVM predictions
plot_confusion_matrix(y_test_decoded, y_pred_svm_decoded, color="Greens")


# K-Fold Cross-Validation

In [ ]:
# Cross-validation must fit the vectorizer separately within each fold.
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
cv_text = reviews.loc[train_mask, 'text']
cv_labels = reviews.loc[train_mask, 'polarity']
for name, classifier in [('Naive Bayes', MultinomialNB()), ('SVM', LinearSVC())]:
    pipeline = make_pipeline(CountVectorizer(min_df=2), classifier)
    scores = cross_validate(pipeline, cv_text, cv_labels, cv=cv,
                            scoring=['accuracy', 'f1_macro'])
    print(name, 'accuracy:', scores['test_accuracy'].mean().round(3),
          'macro F1:', scores['test_f1_macro'].mean().round(3))


In [ ]:


def loo_cross_validation(model, X, y):
    loo = LeaveOneOut()
    y_true_all, y_pred_all = [], []

    for train_index, test_index in loo.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # collect
        y_true_all.append(int(y_test[0]))
        y_pred_all.append(int(y_pred[0]))

    # compute metrics once, globally
    precision = precision_score(y_true_all, y_pred_all, zero_division=0)
    recall = recall_score(y_true_all, y_pred_all, zero_division=0)
    f1 = f1_score(y_true_all, y_pred_all, zero_division=0)
    accuracy = accuracy_score(y_true_all, y_pred_all)

    return {
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Accuracy": accuracy
    }

In [ ]:
dfm_reviews

In [ ]:
# Leave-one-out is computationally expensive for 8,529 training reviews.
# The function above remains available for a small demonstration dataset.
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(reviews['polarity'])
print('Use the five-fold pipeline above for the full corpus.')


## Cross-validated predictions

Five-fold predictions below use training texts only. Each fold learns its own vocabulary; the final test set remains untouched.

In [ ]:
# Out-of-fold predictions on the training data, with a fresh vocabulary per fold.
from sklearn.model_selection import cross_val_predict
for name, classifier in [('Naive Bayes', MultinomialNB()), ('SVM', LinearSVC())]:
    pipeline = make_pipeline(CountVectorizer(min_df=2), classifier)
    pred = cross_val_predict(pipeline, cv_text, cv_labels, cv=cv)
    print(name, classification_report(cv_labels, pred, zero_division=0))


# Area Under PR-Curve

In [ ]:
# Group tied scores correctly using scikit-learn's implementation.
from sklearn.metrics import precision_recall_curve


In [ ]:

# Naive Bayes
y_scores_nb = model_nb.predict_proba(dfm_test)[:, 1]  # P(class=1)
y_true = y_test

precision_nb, recall_nb, _ = precision_recall_curve(y_true, y_scores_nb)
auc_pr_nb = auc(recall_nb, precision_nb)
avg_prec_nb = average_precision_score(y_true, y_scores_nb)

print(f"Naive Bayes AUC-PR (integral): {auc_pr_nb:.3f}")
print(f"Naive Bayes Average Precision: {avg_prec_nb:.3f}")

# SVM 
y_scores_svm = model_svm.decision_function(dfm_test)
precision_svm, recall_svm, _ = precision_recall_curve(y_true, y_scores_svm)
auc_pr_svm = auc(recall_svm, precision_svm)
avg_prec_svm = average_precision_score(y_true, y_scores_svm)

print(f"SVM AUC-PR (integral): {auc_pr_svm:.3f}")
print(f"SVM Average Precision: {avg_prec_svm:.3f}")


plt.figure(figsize=(7,5))
plt.plot(recall_nb, precision_nb, label=f"Naive Bayes (AUC-PR={auc_pr_nb:.2f})")
plt.plot(recall_svm, precision_svm, label=f"SVM (AUC-PR={auc_pr_svm:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Base Python NB Classifier

In [ ]:

class NaiveBayesClassifier:
    def __init__(self):
        self.classes = set()  # Set of unique classes
        self.word_counts = defaultdict(lambda: defaultdict(int))  # Word frequency per class
        self.class_counts = defaultdict(int)  # Total word count per class
        self.total_docs = 0  # Total number of documents
        self.doc_counts = defaultdict(int)  # Document count per class
    
    def train(self, X, y):
        """
        Train the Naive Bayes classifier.
        
        Parameters:
            X (list of lists): List of documents, each represented as a list of words.
            y (list): Corresponding class labels for each document.
        """
        self.total_docs = len(y)
        self.classes = set(y)
        
        for text, label in zip(X, y):
            self.doc_counts[label] += 1
            for word in text:
                self.word_counts[label][word] += 1
                self.class_counts[label] += 1
    
    def predict(self, text):
        """
        Predict the class for a given text.
        
        Parameters:
            text (list): A document represented as a list of words.
        
        Returns:
            str: The predicted class label.
        """
        log_probs = {}
        total_words = sum(self.class_counts.values())
        
        for c in self.classes:
            # Log of the prior probability P(Class)
            log_prior = math.log(self.doc_counts[c] / self.total_docs)
            
            # Log of the likelihood P(Text|Class)
            log_likelihood = 0
            for word in text:
                word_count = self.word_counts[c][word] + 1  # Add-1 smoothing
                class_word_total = self.class_counts[c] + total_words  # Total words in class + smoothing
                log_likelihood += math.log(word_count / class_word_total)
            
            log_probs[c] = log_prior + log_likelihood
        
        # Return the class with the highest posterior probability
        return max(log_probs, key=log_probs.get)


In [ ]:

# Step 1: Preprocess the text
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Tokenize, remove punctuation and stop words, and lowercase
    tokens = word_tokenize(text)
    tokens = [word.lower() for word in tokens if word.isalpha()]  # Keep only alphabetic tokens
    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
    return tokens

# Apply preprocessing
reviews['processed_text'] = reviews['text'].apply(preprocess)

# Step 2: Split the data into training and testing sets
X = reviews['processed_text'].tolist()  # List of tokenized documents
y = reviews['polarity'].tolist()        # List of labels

X_train = [x for x, keep in zip(X, train_mask) if keep]
X_test = [x for x, keep in zip(X, test_mask) if keep]
y_train = [label for label, keep in zip(y, train_mask) if keep]
y_test = [label for label, keep in zip(y, test_mask) if keep]

# Step 3: Initialize and train the classifier
nb_classifier = NaiveBayesClassifier()
nb_classifier.train(X_train, y_train)

# Step 4: Make predictions
y_pred = [nb_classifier.predict(text) for text in X_test]

# Step 5: Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


## Model Complexity and Overfitting

We vary the regularization setting C in logistic regression. Smaller C means stronger regularization. Compare training and validation accuracy rather than assuming that a particular C range always gives a good fit.

Split the raw texts first, then fit TF-IDF and the document-frequency filter on training text only. The fitted representation is used to transform validation text.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

regularization_labels = (reviews["polarity"] == "positive").astype(int).to_numpy()
regularization_train_texts, regularization_val_texts, y_train, y_val = train_test_split(
    reviews.loc[train_mask, "text"].astype(str), regularization_labels[train_mask],
    test_size=0.2, random_state=42, stratify=regularization_labels[train_mask],
)
regularization_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b[a-z]+\b", min_df=2,
)
X_train = regularization_vectorizer.fit_transform(regularization_train_texts)
X_val = regularization_vectorizer.transform(regularization_val_texts)

C_values = np.logspace(-3, 2, 10)
train_scores = []
val_scores = []

for C in C_values:
    model = LogisticRegression(max_iter=2000, C=C, solver="lbfgs")
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    val_scores.append(model.score(X_val, y_val))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(C_values, train_scores, marker="o", label="Training", color="steelblue")
ax.plot(C_values, val_scores, marker="o", label="Validation", color="indianred")
ax.set_xscale("log")
ax.set(xlabel="C: larger means weaker regularization", ylabel="Accuracy",
       title="Logistic regression: training and validation")
ax.tick_params(labelsize=12)
ax.xaxis.label.set_size(13)
ax.yaxis.label.set_size(13)
ax.title.set_size(15)
ax.legend(frameon=False, fontsize=12)
fig.tight_layout()
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/regularization.png", dpi=200)
plt.show()


In [ ]:
best_index = int(np.argmax(val_scores))
print(f"Selected by validation accuracy: C={C_values[best_index]:.3g}")
print(f"Training accuracy={train_scores[best_index]:.3f}")
print(f"Validation accuracy={val_scores[best_index]:.3f}")
print("This is a model-development comparison. Final performance needs a separate test set.")


## Regularization: L1 vs L2 (Sparse vs Smooth Models)

L1 regularization (Lasso) → pushes many coefficients to exactly zero → *sparse* model.
L2 regularization (Ridge) → shrinks coefficients smoothly → *dense* model.

Both control overfitting, but they affect interpretability differently.


In [ ]:
# Fit the representation on this training subset, not its validation subset.
X_train_text, X_val_text, y_train, y_val = train_test_split(
    reviews.loc[train_mask, 'text'], y_encoded[train_mask], test_size=0.2,
    random_state=42, stratify=y_encoded[train_mask])
l1l2_vectorizer = CountVectorizer(min_df=2)
X_train = l1l2_vectorizer.fit_transform(X_train_text)
X_val = l1l2_vectorizer.transform(X_val_text)
model_l1 = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, C=1.0)
model_l2 = LogisticRegression(penalty='l2', solver='liblinear', max_iter=1000, C=1.0)
model_l1.fit(X_train, y_train)
model_l2.fit(X_train, y_train)
print('L1 Accuracy:', round(model_l1.score(X_val, y_val), 3))
print('L2 Accuracy:', round(model_l2.score(X_val, y_val), 3))

In [ ]:
coef_l1 = model_l1.coef_.flatten()
coef_l2 = model_l2.coef_.flatten()

n_nonzero_l1 = np.sum(coef_l1 != 0)
n_nonzero_l2 = np.sum(coef_l2 != 0)

print(f"L1 Non-zero Coefficients: {n_nonzero_l1} / {len(coef_l1)}")
print(f"L2 Non-zero Coefficients: {n_nonzero_l2} / {len(coef_l2)}")

In [ ]:
# %%
plt.figure(figsize=(8,5))
plt.hist(coef_l1, bins=50, alpha=0.6, label='L1 (lasso)', color='firebrick')
plt.hist(coef_l2, bins=50, alpha=0.6, label='L2 (ridge)', color='steelblue')
plt.xlabel('Coefficient Value')
plt.ylabel('Frequency')
plt.title('Distribution of Logistic Regression Coefficients')
plt.legend()
plt.show()

In [ ]:
feature_names = np.array(l1l2_vectorizer.get_feature_names_out())

top_pos_l1 = feature_names[np.argsort(coef_l1)[-10:]]
top_neg_l1 = feature_names[np.argsort(coef_l1)[:10]]

top_pos_l2 = feature_names[np.argsort(coef_l2)[-10:]]
top_neg_l2 = feature_names[np.argsort(coef_l2)[:10]]

print("Top positive words (L1):", ", ".join(top_pos_l1))
print("Top negative words (L1):", ", ".join(top_neg_l1))
print("Top positive words (L2):", ", ".join(top_pos_l2))
print("Top negative words (L2):", ", ".join(top_neg_l2))

In [ ]:
C_values = np.logspace(-2, 1, 8)
nonzero_counts = []
accuracies = []

for C in C_values:
    model = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, C=C)
    model.fit(X_train, y_train)
    nonzero_counts.append(np.sum(model.coef_.flatten() != 0))
    accuracies.append(model.score(X_val, y_val))

fig, ax1 = plt.subplots(figsize=(8,5))
ax1.plot(C_values, accuracies, marker='o', color='darkorange', label='Accuracy')
ax1.set_xscale('log')
ax1.set_xlabel('C (inverse regularization strength)')
ax1.set_ylabel('Validation Accuracy', color='darkorange')
ax2 = ax1.twinx()
ax2.plot(C_values, nonzero_counts, marker='s', color='steelblue', label='Non-zero Coefficients')
ax2.set_ylabel('Active Features', color='steelblue')
plt.title('L1 Regularization: Sparsity vs. Accuracy')
plt.show()

In [ ]:
theta1, theta2 = np.meshgrid(np.linspace(-2, 2, 400), np.linspace(-2, 2, 400))
loss = theta1**2 / 2 + 2*(theta2 - 0.5)**2 / 3  # elliptical contour

plt.figure(figsize=(5,5))
# Contours of loss
plt.contour(theta1, theta2, loss, levels=10, cmap='Greys', alpha=0.7)

# L2 constraint (circle)
circle = plt.Circle((0,0), 1.3, color='C0', fill=False, lw=2, label='L2 constraint')

# L1 constraint (diamond)
diamond = np.array([[0,1.3],[1.3,0],[0,-1.3],[-1.3,0],[0,1.3]])
plt.plot(diamond[:,0], diamond[:,1], color='C1', lw=2, label='L1 constraint')

plt.gca().add_patch(circle)
plt.axhline(0, color='gray', lw=1)
plt.axvline(0, color='gray', lw=1)
plt.xlabel(r'$\theta_1$')
plt.ylabel(r'$\theta_2$')
plt.legend()
plt.title("L1 vs L2 Regularization")
plt.grid(alpha=0.3)
plt.axis('equal')
plt.show()

# SHAP / LIME

Shapley values (from game theory) / Local Interpretable Model-agnostic Explanations

In [ ]:

# Prepare raw text and labels
texts = reviews["text"]
labels = reviews["polarity"]

# Encode labels
from sklearn.preprocessing import LabelEncoder
label_enc = LabelEncoder()
y = label_enc.fit_transform(labels)

X_train, X_test = texts[train_mask], texts[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(X_train_tfidf, y_train)

print("Test accuracy:", clf.score(X_test_tfidf, y_test))

In [ ]:
# LIME: Local word-level explanation (fixed version)

explainer = LimeTextExplainer(class_names=label_enc.classes_)

# Pick an example review
i = 666
text_instance = X_test.iloc[i]
print("Text:", text_instance)
print("True label:", label_enc.classes_[y_test[i]])

# Wrapper for vectorizer + model
def predict_proba_wrapper(texts):
    return clf.predict_proba(tfidf.transform(texts))

exp = explainer.explain_instance(
    text_instance,
    predict_proba_wrapper,
    num_features=10,
    labels=[0, 1]
)

from IPython.display import HTML, display
display(HTML(exp.as_html()))

In [ ]:
# Shap

feature_names = tfidf.get_feature_names_out()

explainer = shap.LinearExplainer(
    clf,
    X_train_tfidf,
    feature_perturbation="interventional",
    feature_names=feature_names
)

shap_values = explainer(X_test_tfidf)

In [ ]:
feature_names = tfidf.get_feature_names_out()
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

importance_df = (
    pd.DataFrame({"word": feature_names, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .head(50)  # show top 50 words
)

plt.figure(figsize=(8, 10))
plt.barh(importance_df["word"], importance_df["mean_abs_shap"])
plt.gca().invert_yaxis()
plt.xlabel("Mean |SHAP Value|")
plt.title("Top 50 Most Important Words (Global SHAP Importance)")
plt.show()

In [ ]:
feature_names = tfidf.get_feature_names_out()
shap_matrix = shap_values.values  # shape: (n_samples, n_features)

# Mean SHAP value across all samples (signed)
mean_shap = shap_matrix.mean(axis=0)

# Separate top positive and top negative words
top_pos = (
    pd.DataFrame({"word": feature_names, "mean_shap": mean_shap})
    .sort_values("mean_shap", ascending=False)
    .head(10)
)

top_neg = (
    pd.DataFrame({"word": feature_names, "mean_shap": mean_shap})
    .sort_values("mean_shap", ascending=True)
    .head(10)
)

print("Top positive words:\n", top_pos)
print("\nTop negative words:\n", top_neg)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

ax[0].barh(top_neg["word"], top_neg["mean_shap"], color="steelblue")
ax[0].set_title("Words pushing towards Negative sentiment")
ax[0].invert_yaxis()

ax[1].barh(top_pos["word"], top_pos["mean_shap"], color="coral")
ax[1].set_title("Words pushing towards Positive sentiment")

plt.tight_layout()
plt.show()

In [ ]:
shap.plots.beeswarm(shap_values)

## Putting it together: a complete Naive Bayes workflow

Having seen the individual steps, we now compare a dictionary and multinomial Naive Bayes on the same reviews. We use five-fold cross-validation on the training set to select the smoothing parameter. The vocabulary is learned separately inside each fold, and the selected model is refit on the training set before test evaluation.

The earlier teaching examples also showed results on this split. For an independent final assessment after further experimentation, set aside a new, previously uninspected test set.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve as sklearn_pr_curve, f1_score,
                             accuracy_score, precision_score, recall_score)
import re

texts = reviews['text'].astype(str)
labels = (reviews['polarity'] == 'positive').astype(int)
X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    texts, labels, test_size=0.2, random_state=2026, stratify=labels)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
pipeline = Pipeline([('counts', CountVectorizer(min_df=2)),
                     ('model', MultinomialNB())])
search = GridSearchCV(pipeline, {'model__alpha': [0.1, 1.0, 10.0]},
                      scoring='f1', cv=folds, n_jobs=1, return_train_score=True)
search.fit(X_dev, y_dev)
display(pd.DataFrame(search.cv_results_)[['param_model__alpha', 'mean_test_score', 'std_test_score']])
print('Selected setting:', search.best_params_)
print('Training and test sizes:', len(X_dev), len(X_holdout))

### Dictionary and supervised model on the same test set

The dictionary score counts positive matches minus negative matches; a score above zero predicts positive. Ties are negative. This small dictionary is the Class 4 starting point, not a tuned final dictionary.

In [ ]:
positive_terms = {'beautiful', 'best', 'brilliant', 'charming', 'compelling', 'enjoyable', 'entertaining', 'excellent', 'fantastic', 'funny', 'good', 'great', 'moving', 'wonderful'}
negative_terms = {'awful', 'bad', 'boring', 'disappointing', 'dull', 'poor', 'ridiculous', 'stupid', 'terrible', 'waste', 'worst'}
def dictionary_score(text):
    tokens = re.findall(r'[a-z]+', text.lower())
    return sum(t in positive_terms for t in tokens) - sum(t in negative_terms for t in tokens)
dict_scores = X_holdout.map(dictionary_score).to_numpy()
dict_pred = (dict_scores > 0).astype(int)
majority = DummyClassifier(strategy='most_frequent').fit(X_dev, y_dev)
model_pred = search.predict(X_holdout)
model_scores = search.predict_proba(X_holdout)[:, 1]
rows = []
for name, pred in [('Majority', majority.predict(X_holdout)), ('Dictionary', dict_pred), ('Naive Bayes', model_pred)]:
    rows.append({'Method': name, 'Accuracy': accuracy_score(y_holdout, pred),
                 'Precision': precision_score(y_holdout, pred, zero_division=0),
                 'Recall': recall_score(y_holdout, pred, zero_division=0),
                 'F1': f1_score(y_holdout, pred, zero_division=0)})
display(pd.DataFrame(rows).set_index('Method').round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name, pred in zip(axes, ['Dictionary', 'Naive Bayes'], [dict_pred, model_pred]):
    ConfusionMatrixDisplay.from_predictions(y_holdout, pred, display_labels=['Negative', 'Positive'], ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, scores in [('Dictionary', dict_scores), ('Naive Bayes', model_scores)]:
    precision, recall, thresholds = sklearn_pr_curve(y_holdout, scores)
    ax.step(recall, precision, where='post', label=name)
ax.axhline(y_holdout.mean(), color='grey', linestyle='--', label='Positive prevalence')
ax.set(xlabel='Recall', ylabel='Precision', title='Held-out precision-recall curves')
ax.legend(); plt.show()
# Thresholds describe trade-offs; do not tune a final rule on this test set.

### Inspect errors and learned features

These examples show what each method gets wrong. The review-level source label may not match the sentiment of every isolated snippet. Feature weights describe predictive associations rather than causal effects.

For Naive Bayes, the word weights below are differences in log word probabilities between the positive and negative classes.

In [ ]:
for name, pred, scores in [('Dictionary', dict_pred, dict_scores), ('Naive Bayes', model_pred, model_scores)]:
    errors = pd.DataFrame({'text': X_holdout.to_numpy(), 'reference': y_holdout.to_numpy(),
                           'prediction': pred, 'score': scores})
    for label, truth, prediction in [('False positives', 0, 1), ('False negatives', 1, 0)]:
        print(name, label)
        display(errors[(errors.reference == truth) & (errors.prediction == prediction)].head(2))
features = search.best_estimator_.named_steps['counts'].get_feature_names_out()
nb = search.best_estimator_.named_steps['model']
# Difference in log word probabilities, positive minus negative.
weights = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
print('Most negative:', list(zip(features[np.argsort(weights)[:10]], np.sort(weights)[:10].round(3))))
print('Most positive:', list(zip(features[np.argsort(weights)[-10:][::-1]], np.sort(weights)[-10:][::-1].round(3))))